# Attention（注意機構）の基礎

RNN/LSTM の限界を克服するために生まれた Attention の仕組みを、ゼロから理解して実装します。

## 学習の流れ
1. なぜ Attention が必要か
2. Attention の計算を手動で理解
3. Seq2Seq + Attention の実装
4. Attention の重みを可視化
5. Self-Attention（Transformer への橋渡し）

## 1. なぜ Attention が必要か

### Encoder-Decoder の問題点

LSTM ベースの Encoder-Decoder モデルでは、入力系列全体を**固定長の隠れ状態ベクトル1つ**に圧縮します。

```
入力: [x1, x2, x3, ..., x100]
          ↓ LSTM Encoder
    固定長ベクトル h（1つだけ）
          ↓ LSTM Decoder
出力: [y1, y2, y3, ...]
```

**問題**: 入力が長くなると、1つのベクトルに全情報を詰め込むのが困難（**ボトルネック**）。

sin波の実験で見たように、LSTM は長い系列で情報が薄れていきます。

### Attention のアイデア

Decoder が各出力を生成するとき、Encoder の**全時刻の出力を参照**し、
**どの時刻に注目すべきか**を動的に決める。

```
入力: [x1, x2, x3, ..., x100]
          ↓ LSTM Encoder
    [h1, h2, h3, ..., h100]  ← 全時刻の出力を保持
          ↓ Attention で動的に重み付け
    Context vector（出力ごとに異なる）
          ↓ LSTM Decoder
出力: [y1, y2, y3, ...]
```

固定長ベクトル1つではなく、**必要な情報がある場所を毎回選んで参照**できる。

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
print('PyTorch version:', torch.__version__)

## 2. Attention の仕組み

Attention の核心は **3つのステップ**:

1. **スコア計算**: Decoder の状態と Encoder の各出力の「関連度」を計算
2. **正規化**: Softmax でスコアを確率分布に変換 → **Attention weights**
3. **重み付き和**: Encoder 出力の加重平均 → **Context vector**

### Query, Key, Value

| 役割 | 意味 | 対応するもの |
|------|------|-------------|
| **Query (Q)** | 「何を探しているか」 | Decoder の現在の隠れ状態 |
| **Key (K)** | 「何があるか」 | Encoder の各時刻の出力 |
| **Value (V)** | 「実際の情報」 | Encoder の各時刻の出力 |

スコア = Query と Key の類似度（内積）

In [ ]:
# === Attention の計算をステップバイステップで理解する ===

# Encoder の各時刻の出力（Key & Value）
# 例: 4つの単語、各4次元のベクトル
encoder_outputs = np.array([
    [1.0, 0.0, 1.0, 0.0],  # h0: "I"
    [0.0, 1.0, 0.0, 1.0],  # h1: "love"
    [0.5, 0.5, 0.5, 0.5],  # h2: "cute"
    [1.0, 1.0, 0.0, 0.0],  # h3: "cats"
])

# Decoder の現在の隠れ状態（Query）
# "猫" を出力しようとしている想定
decoder_hidden = np.array([0.9, 0.9, 0.1, 0.1])

# --- Step 1: スコア計算（内積）---
print('=== Step 1: スコア計算（内積）===')
scores = encoder_outputs @ decoder_hidden
for s, label in zip(scores, ['I', 'love', 'cute', 'cats']):
    print(f'  {label:>4s}: {s:.2f}')

# --- Step 2: Softmax で正規化 ---
print('\n=== Step 2: Softmax → Attention weights ===')
exp_scores = np.exp(scores - scores.max())
attention_weights = exp_scores / exp_scores.sum()
for w, label in zip(attention_weights, ['I', 'love', 'cute', 'cats']):
    print(f'  {label:>4s}: {w:.3f}')
print(f'  合計: {attention_weights.sum():.1f}')

# --- Step 3: 重み付き和 → Context vector ---
print('\n=== Step 3: 重み付き和 → Context vector ===')
context = attention_weights @ encoder_outputs
print(f'  Context: [{" ".join(f"{v:.3f}" for v in context)}]')
print('  → "cats" と "I" に注目が集中し、それらの情報が Context に反映される')

# --- 可視化 ---
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
labels = ['I', 'love', 'cute', 'cats']
colors = ['#4CAF50', '#2196F3', '#FF9800', '#E91E63']

axes[0].bar(labels, attention_weights, color=colors)
axes[0].set_ylabel('Attention Weight')
axes[0].set_title('Attention Weights')
axes[0].set_ylim(0, 1)

im = axes[1].imshow(attention_weights.reshape(1, -1), cmap='Blues', aspect='auto')
axes[1].set_xticks(range(4))
axes[1].set_xticklabels(labels)
axes[1].set_yticks([0])
axes[1].set_yticklabels(['Query'])
axes[1].set_title('Attention Heatmap')
for i, w in enumerate(attention_weights):
    axes[1].text(i, 0, f'{w:.3f}', ha='center', va='center', fontsize=11)
plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.savefig('attention_basics_step.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Seq2Seq + Attention の実装

### Scaled Dot-Product Attention

実際の Attention では、内積をベクトルの次元数 $d_k$ の平方根で割ります（スケーリング）。

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

次元数が大きいと内積の値が大きくなり、Softmax が極端な分布になって勾配消失が起きるため。

### タスク: 系列の反転

入力 `[3, 1, 4, 1, 5, 9]` → 出力 `[9, 5, 1, 4, 1, 3]`

Attention が入力の**どの位置**に注目するかを観察できます。
反転タスクでは、Attention weights が逆対角線パターンになるはずです。

### モデル構成

```
入力 → Embedding → LSTM Encoder → [h1, h2, ..., hn]（全時刻の出力）
                                          ↓ Attention
                                   Context vector
                                          ↓
                              LSTM Decoder → 出力
```

In [ ]:
# === 定数 ===
VOCAB_SIZE = 10   # 数字 0-9
SOS_TOKEN = 10    # Start of Sequence
NUM_TOKENS = 11   # 0-9 + SOS
HIDDEN_SIZE = 64
SEQ_LEN = 6
BATCH_SIZE = 64


# === Encoder ===
class Encoder(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedded)
        # outputs: 全時刻の隠れ状態 (batch, seq_len, hidden)
        return outputs, hidden, cell


# === Attention 付き Decoder (Bahdanau Attention) ===
class AttentionDecoder(nn.Module):
    """
    Bahdanau (Additive) Attention:
      score = v^T * tanh(W_a[decoder_hidden; encoder_output])
    
    Dot-Product との違い:
      - Dot-Product: score = decoder_hidden^T * encoder_output（単純な内積）
      - Bahdanau: 学習可能な重み W_a を使い、より柔軟にスコアを計算
    """
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        # Attention 層
        self.W_a = nn.Linear(hidden_size * 2, hidden_size)
        self.v_a = nn.Linear(hidden_size, 1, bias=False)
        # Decoder LSTM（入力 = embedded + context）
        self.lstm = nn.LSTM(hidden_size * 2, hidden_size, batch_first=True)
        self.fc_out = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden, cell, encoder_outputs):
        embedded = self.embedding(x)  # (batch, 1, hidden)

        # --- Attention ---
        seq_len = encoder_outputs.size(1)
        # Decoder の前の隠れ状態を各時刻分コピー
        h = hidden[-1].unsqueeze(1).expand(-1, seq_len, -1)
        # Encoder出力と結合してスコア計算
        energy = torch.tanh(self.W_a(torch.cat([h, encoder_outputs], dim=-1)))
        scores = self.v_a(energy).squeeze(-1)          # (batch, seq_len)
        attention_weights = F.softmax(scores, dim=-1)   # (batch, seq_len)

        # Context vector（Encoder 出力の重み付き和）
        context = torch.bmm(attention_weights.unsqueeze(1), encoder_outputs)

        # --- Decoder LSTM ---
        lstm_input = torch.cat([embedded, context], dim=-1)
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))

        prediction = self.fc_out(output.squeeze(1))  # (batch, vocab_size)
        return prediction, hidden, cell, attention_weights


# === Seq2Seq モデル ===
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        trg_len = trg.size(1)

        outputs = torch.zeros(batch_size, trg_len, NUM_TOKENS)
        attentions = torch.zeros(batch_size, trg_len, src.size(1))

        encoder_outputs, hidden, cell = self.encoder(src)

        # 最初の入力は SOS トークン
        decoder_input = torch.full((batch_size, 1), SOS_TOKEN, dtype=torch.long)

        for t in range(trg_len):
            prediction, hidden, cell, attn_w = self.decoder(
                decoder_input, hidden, cell, encoder_outputs
            )
            outputs[:, t] = prediction
            attentions[:, t] = attn_w

            # Teacher Forcing: 一定確率で正解を次の入力にする
            if torch.rand(1).item() < teacher_forcing_ratio:
                decoder_input = trg[:, t].unsqueeze(1)
            else:
                decoder_input = prediction.argmax(dim=-1).unsqueeze(1)

        return outputs, attentions


encoder = Encoder(NUM_TOKENS, HIDDEN_SIZE)
decoder = AttentionDecoder(NUM_TOKENS, HIDDEN_SIZE)
model = Seq2Seq(encoder, decoder)
total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')

In [ ]:
# === データ生成 ===
def generate_data(n_samples, seq_len):
    """系列反転データ: [3,1,4,1,5,9] -> [9,5,1,4,1,3]"""
    src = torch.randint(0, VOCAB_SIZE, (n_samples, seq_len))
    trg = src.flip(dims=[1])
    return src, trg

train_src, train_trg = generate_data(3000, SEQ_LEN)
test_src, test_trg = generate_data(500, SEQ_LEN)

print(f'Train: {len(train_src)} samples')
print(f'Test:  {len(test_src)} samples')
print(f'Example: {train_src[0].tolist()} -> {train_trg[0].tolist()}')

# === 学習 ===
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

losses = []
for epoch in range(100):
    model.train()
    epoch_loss = 0
    n_batches = 0

    indices = torch.randperm(len(train_src))
    for i in range(0, len(train_src), BATCH_SIZE):
        batch_idx = indices[i:i+BATCH_SIZE]
        src = train_src[batch_idx]
        trg = train_trg[batch_idx]

        optimizer.zero_grad()
        output, _ = model(src, trg, teacher_forcing_ratio=0.5)
        loss = criterion(output.reshape(-1, NUM_TOKENS), trg.reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)

    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1:3d}/100  Loss: {avg_loss:.4f}')

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True, alpha=0.3)
plt.savefig('attention_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === 評価（完全自己回帰: 自分の予測を次の入力に使う）===
model.eval()
correct_count = 0
n_eval = 100
first_attn = None
first_src = None
first_pred = None

with torch.no_grad():
    for i in range(n_eval):
        src = test_src[i:i+1]
        trg = test_trg[i:i+1]

        encoder_outputs, hidden, cell = model.encoder(src)
        decoder_input = torch.full((1, 1), SOS_TOKEN, dtype=torch.long)

        predicted = []
        attn_list = []

        for t in range(SEQ_LEN):
            pred, hidden, cell, attn_w = model.decoder(
                decoder_input, hidden, cell, encoder_outputs
            )
            token = pred.argmax(dim=-1).item()
            predicted.append(token)
            attn_list.append(attn_w.squeeze().numpy())
            decoder_input = torch.tensor([[token]])

        target = trg.squeeze().tolist()
        is_correct = (predicted == target)
        if is_correct:
            correct_count += 1

        # 最初の5件を表示
        if i < 5:
            mark = 'OK' if is_correct else 'NG'
            print(f'Input:  {src.squeeze().tolist()}')
            print(f'Target: {target}')
            print(f'Pred:   {predicted}  [{mark}]')
            print()

        # 最初の例の Attention を保存
        if i == 0:
            first_attn = np.array(attn_list)
            first_src = src.squeeze().tolist()
            first_pred = predicted

print(f'Sequence Accuracy: {correct_count}/{n_eval} = {correct_count/n_eval*100:.1f}%')

In [ ]:
# === Attention Heatmap ===
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(first_attn, cmap='Blues', aspect='auto')

src_labels = [str(x) for x in first_src]
pred_labels = [str(x) for x in first_pred]

ax.set_xticks(range(SEQ_LEN))
ax.set_yticks(range(SEQ_LEN))
ax.set_xticklabels(src_labels, fontsize=12)
ax.set_yticklabels(pred_labels, fontsize=12)
ax.set_xlabel('Input (Encoder)', fontsize=12)
ax.set_ylabel('Output (Decoder)', fontsize=12)
ax.set_title('Attention Weights - Sequence Reversal', fontsize=14)

for i in range(SEQ_LEN):
    for j in range(SEQ_LEN):
        ax.text(j, i, f'{first_attn[i, j]:.2f}',
                ha='center', va='center', fontsize=9)

plt.colorbar(im)
plt.tight_layout()
plt.savefig('attention_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('反転タスク → 逆対角線パターン（右上→左下）になるはず')
print('出力1番目 → 入力の最後に注目')
print('出力2番目 → 入力の最後から2番目に注目 ...')

## 4. Self-Attention（自己注意）

ここまでの Attention は **Encoder と Decoder の間**で使いました。

**Self-Attention** は、**同じ系列内**の各要素が他の全要素との関連度を計算します。

```
通常の Attention:  Decoder(Query) → Encoder(Key, Value)
Self-Attention:    同じ入力から Query, Key, Value を全て生成
```

### なぜ重要か

- Transformer は Self-Attention だけで構成される（RNN/LSTM を使わない）
- 系列内の**任意の2要素間の関係**を直接学習できる
- LSTM のように順番に処理する必要がなく、**並列計算**が可能

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$
$$\text{Self-Attention}(X) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
class SelfAttention(nn.Module):
    """各要素が同じ系列内の他の全要素との関連度を計算"""
    def __init__(self, d_model):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.d_model = d_model

    def forward(self, x):
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_model ** 0.5)
        attention_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attention_weights, V)
        return output, attention_weights


# デモ: 5単語の Self-Attention
self_attn = SelfAttention(d_model=16)
torch.manual_seed(0)
x = torch.randn(1, 5, 16)  # (batch=1, 5 tokens, dim=16)
words = ['I', 'love', 'my', 'cute', 'cats']

with torch.no_grad():
    _, weights = self_attn(x)

w = weights.squeeze().numpy()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(w, cmap='Blues')
ax.set_xticks(range(5))
ax.set_yticks(range(5))
ax.set_xticklabels(words, fontsize=11)
ax.set_yticklabels(words, fontsize=11)
ax.set_xlabel('Key (attend to)', fontsize=12)
ax.set_ylabel('Query (from)', fontsize=12)
ax.set_title('Self-Attention Weights', fontsize=14)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f'{w[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im)
plt.tight_layout()
plt.savefig('self_attention.png', dpi=150, bbox_inches='tight')
plt.show()

print('各単語が他の全単語との関連度を持つ（5x5 の行列）')
print('学習前なのでランダムに近いが、学習後は意味的に関連する単語間の重みが高くなる')
print()
print('例: "cats" の行 → "cute" や "love" に高い重みがつくようになる')
print('Transformer はこの Self-Attention を何層も積み重ねて構築される')

## まとめ

| 概念 | 要点 |
|------|------|
| **Attention** | Decoder が Encoder のどこに注目するかを動的に決定 |
| **QKV** | Query（何を探す）× Key（何がある）→ 重み → Value（情報）の加重和 |
| **Bahdanau** | 学習可能な重みでスコア計算（Additive Attention） |
| **Scaled Dot-Product** | 内積÷√d_k でスコア計算（Transformer で使用） |
| **Self-Attention** | 同じ系列内の全要素間の関連度を計算 |

### 学習の流れ

```
RNN → LSTM → Attention → Transformer
 ✅     ✅       ✅         次はここ
```

次のステップ: **Transformer** の実装
- Self-Attention を複数層積む（Multi-Head Attention）
- Positional Encoding（位置情報の付与）
- RNN/LSTM を完全に排除した構造